# Stability-Guided Ensemble Pruning for Improved Generalization

This notebook implements a stability-guided ensemble pruning pipeline that:
1. Evaluates individual classifier stability via repeated cross-validation
2. Prunes unstable/weak models based on ROC-AUC mean and variance
3. Compares pruned ensembles against full ensembles and individual models
4. Tunes only the pruned subset for final performance

**Datasets:** Heart Disease (UCI) and Pima Indians Diabetes

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier

from ucimlrepo import fetch_ucirepo

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All imports successful.')

All imports successful.


## 1. Load Datasets

In [2]:
# --- Heart Disease (UCI) ---
heart_repo = fetch_ucirepo(id=45)
heart_X = heart_repo.data.features.copy()
heart_y = heart_repo.data.targets.copy()

# The target have values 0-4; binarise: 0 = no disease, 1 = disease
heart_y.columns = ['target']
heart_y['target'] = (heart_y['target'] > 0).astype(int)

heart_df = pd.concat([heart_X, heart_y], axis=1)
heart_df.to_csv('heart_disease_clean.csv', index=False)
print(f'Heart Disease shape: {heart_df.shape}')
print(f'Class distribution:\n{heart_df["target"].value_counts()}')
print(f'Features: {list(heart_X.columns)}')
print()

Heart Disease shape: (303, 14)
Class distribution:
target
0    164
1    139
Name: count, dtype: int64
Features: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']



In [3]:
# --- Pima Indians Diabetes ---
pima_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
             'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
pima_df = pd.read_csv(
    'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv',
    header=None, names=pima_cols
)
pima_df.to_csv('pima_diabetes_clean.csv', index=False)
print(f'Pima Diabetes shape: {pima_df.shape}')
print(f'Class distribution:\n{pima_df["Outcome"].value_counts()}')
print(f'Features: {pima_cols[:-1]}')
print()

Pima Diabetes shape: (768, 9)
Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64
Features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']



## 2. Preprocessing

We handle missing values via median imputation inside a Pipeline together with StandardScaler.
Scaling is performed **inside each CV fold** (never on the full dataset) to prevent data leakage.

In [4]:
# Prepare X, y for each dataset
# Heart Disease
heart_X = heart_df.drop('target', axis=1)
heart_y = heart_df['target'].values

# Replace any zero values in columns where 0 is physiologically impossible with NaN
# (Heart dataset may have missing values encoded as NaN already from UCI)
print('Heart Disease missing values per column:')
print(heart_X.isnull().sum())
print()

# Pima Diabetes — zeros in Glucose, BloodPressure, SkinThickness, Insulin, BMI are missing
pima_X = pima_df.drop('Outcome', axis=1).copy()
pima_y = pima_df['Outcome'].values

zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
pima_X[zero_cols] = pima_X[zero_cols].replace(0, np.nan)
print('Pima Diabetes missing values per column (after replacing 0s):')
print(pima_X.isnull().sum())
print()

# Check class balance
for name, y in [('Heart', heart_y), ('Pima', pima_y)]:
    counts = np.bincount(y)
    ratio = counts.min() / counts.max()
    print(f'{name}: class 0 = {counts[0]}, class 1 = {counts[1]}, minority ratio = {ratio:.2f}')
    if ratio < 0.5:
        print(f'  -> Imbalanced! Will include PR-AUC in evaluation.')
    else:
        print(f'  -> Reasonably balanced.')
print()

Heart Disease missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
dtype: int64

Pima Diabetes missing values per column (after replacing 0s):
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
dtype: int64

Heart: class 0 = 164, class 1 = 139, minority ratio = 0.85
  -> Reasonably balanced.
Pima: class 0 = 500, class 1 = 268, minority ratio = 0.54
  -> Reasonably balanced.



## 3. Define Classifier Pool (Default Hyperparameters)

All classifiers use **default** hyperparameters. This is intentional to ensure a fair, unbiased,
leakage-free stability comparison across all models.

In [5]:
def get_base_classifiers():
    """Return dict of name -> classifier with default hyperparameters."""
    return {
        'LR': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        'SVM': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
        'kNN': KNeighborsClassifier(n_neighbors=5),
        'RF': RandomForestClassifier(random_state=RANDOM_STATE),
        'XGBoost': XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss',
                                  use_label_encoder=False)
    }

def make_pipeline(clf):
    """Wrap classifier in Imputer + Scaler pipeline."""
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', clf)
    ])

print('Classifier pool defined: LR, SVM, kNN, RF, XGBoost')

Classifier pool defined: LR, SVM, kNN, RF, XGBoost


## 4. Stage 1 — Stability Evaluation (Default Params)

RepeatedStratifiedKFold (5 splits x 5 repeats = 25 scores per model).  
Scaler is fit **only on training data** per fold via Pipeline.

In [6]:
def evaluate_stability(X, y, dataset_name):
    """Run Stage 1 stability evaluation. Returns results DataFrame and per-fold scores dict."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    scoring = ['accuracy', 'f1_weighted', 'roc_auc']
    
    classifiers = get_base_classifiers()
    results = []
    fold_scores = {}  # name -> array of AUC scores per fold
    
    for name, clf in classifiers.items():
        pipe = make_pipeline(clf)
        cv_results = cross_validate(pipe, X, y, cv=rskf, scoring=scoring,
                                     return_train_score=False, n_jobs=-1)
        1
        acc = cv_results['test_accuracy']
        f1 = cv_results['test_f1_weighted']
        auc = cv_results['test_roc_auc']
        
        stability = auc.mean() / (auc.std() + 1e-9)
        fold_scores[name] = auc
        
        results.append({
            'Model': name,
            'Mean Acc': acc.mean(), 'Std Acc': acc.std(),
            'Mean F1': f1.mean(), 'Std F1': f1.std(),
            'Mean AUC': auc.mean(), 'Std AUC': auc.std(),
            'Stability Score': stability
        })
    
    df = pd.DataFrame(results)
    df.to_csv(f'stability_results_{dataset_name}.csv', index=False)
    
    print(f'\n=== Stability Results — {dataset_name.upper()} ===')
    display_df = df.copy()
    for col in df.columns:
        if col not in ['Model']:
            display_df[col] = df[col].map(lambda x: f'{x:.4f}')
    print(display_df.to_string(index=False))
    
    return df, fold_scores

# Run for both datasets
print('Running stability evaluation for Heart Disease...')
heart_stability_df, heart_fold_scores = evaluate_stability(heart_X, heart_y, 'heart')

print('\nRunning stability evaluation for Pima Diabetes...')
pima_stability_df, pima_fold_scores = evaluate_stability(pima_X, pima_y, 'pima')

Running stability evaluation for Heart Disease...



=== Stability Results — HEART ===
  Model Mean Acc Std Acc Mean F1 Std F1 Mean AUC Std AUC Stability Score
     LR   0.8375  0.0424  0.8365 0.0426   0.9059  0.0391         23.1836
    SVM   0.8342  0.0382  0.8331 0.0385   0.8990  0.0378         23.7678
    kNN   0.8303  0.0375  0.8295 0.0379   0.8907  0.0387         23.0077
     RF   0.8315  0.0496  0.8302 0.0502   0.9047  0.0369         24.4944
XGBoost   0.8038  0.0464  0.8027 0.0471   0.8787  0.0358         24.5689

Running stability evaluation for Pima Diabetes...

=== Stability Results — PIMA ===
  Model Mean Acc Std Acc Mean F1 Std F1 Mean AUC Std AUC Stability Score
     LR   0.7677  0.0323  0.7594 0.0340   0.8361  0.0303         27.6001
    SVM   0.7622  0.0284  0.7540 0.0296   0.8298  0.0276         30.0302
    kNN   0.7297  0.0410  0.7275 0.0405   0.7891  0.0352         22.4137
     RF   0.7638  0.0326  0.7599 0.0334   0.8287  0.0327         25.3734
XGBoost   0.7440  0.0344  0.7429 0.0344   0.8019  0.0360         22.2496


## 5. Stage 2 — Pruning

Select models where **both**:
- Mean ROC-AUC >= average of all models' Mean AUC
- Std ROC-AUC <= average of all models' Std AUC

If fewer than 2 models selected, relax threshold slightly and retry.

In [7]:
def prune_models(stability_df, dataset_name):
    """Select stable, high-performing models. Returns list of selected model names."""
    mean_auc_threshold = stability_df['Mean AUC'].mean()
    std_auc_threshold = stability_df['Std AUC'].mean()
    
    relaxation = 0.0
    selected = []
    
    while len(selected) < 2:
        auc_thresh = mean_auc_threshold - relaxation
        std_thresh = std_auc_threshold + relaxation
        
        mask = (stability_df['Mean AUC'] >= auc_thresh) & (stability_df['Std AUC'] <= std_thresh)
        selected = stability_df[mask]['Model'].tolist()
        
        if len(selected) < 2:
            relaxation += 0.005
            if relaxation > 0.1:
                # Fallback: pick top 2 by stability score
                selected = stability_df.nlargest(2, 'Stability Score')['Model'].tolist()
                break
    
    print(f'\n=== Pruning Results — {dataset_name.upper()} ===')
    print(f'Mean AUC threshold: {auc_thresh:.4f} (original: {mean_auc_threshold:.4f})')
    print(f'Std AUC threshold:  {std_thresh:.4f} (original: {std_auc_threshold:.4f})')
    if relaxation > 0:
        print(f'(Threshold relaxed by {relaxation:.3f} to ensure >= 2 models selected)')
    print()
    
    for _, row in stability_df.iterrows():
        name = row['Model']
        if name in selected:
            print(f'  SELECTED: {name} — Mean AUC={row["Mean AUC"]:.4f}, Std AUC={row["Std AUC"]:.4f}')
        else:
            reasons = []
            if row['Mean AUC'] < auc_thresh:
                reasons.append(f'Mean AUC {row["Mean AUC"]:.4f} < {auc_thresh:.4f}')
            if row['Std AUC'] > std_thresh:
                reasons.append(f'Std AUC {row["Std AUC"]:.4f} > {std_thresh:.4f}')
            print(f'  REJECTED: {name} — {"; ".join(reasons)}')
    
    return selected

heart_selected = prune_models(heart_stability_df, 'heart')
pima_selected = prune_models(pima_stability_df, 'pima')


=== Pruning Results — HEART ===
Mean AUC threshold: 0.8908 (original: 0.8958)
Std AUC threshold:  0.0427 (original: 0.0377)
(Threshold relaxed by 0.005 to ensure >= 2 models selected)

  SELECTED: LR — Mean AUC=0.9059, Std AUC=0.0391
  SELECTED: SVM — Mean AUC=0.8990, Std AUC=0.0378
  REJECTED: kNN — Mean AUC 0.8907 < 0.8908
  SELECTED: RF — Mean AUC=0.9047, Std AUC=0.0369
  REJECTED: XGBoost — Mean AUC 0.8787 < 0.8908

=== Pruning Results — PIMA ===
Mean AUC threshold: 0.8171 (original: 0.8171)
Std AUC threshold:  0.0324 (original: 0.0324)

  SELECTED: LR — Mean AUC=0.8361, Std AUC=0.0303
  SELECTED: SVM — Mean AUC=0.8298, Std AUC=0.0276
  REJECTED: kNN — Mean AUC 0.7891 < 0.8171; Std AUC 0.0352 > 0.0324
  REJECTED: RF — Std AUC 0.0327 > 0.0324
  REJECTED: XGBoost — Mean AUC 0.8019 < 0.8171; Std AUC 0.0360 > 0.0324


## 6. Stage 3 — Build & Evaluate All Methods

Evaluate 4 method categories (individuals already done in Stage 1):
1. Individual classifiers (reuse Stage 1)
2. Full Soft Voting Ensemble
3. Full Stacking Ensemble
4. Pruned Soft Voting Ensemble (default params)

In [8]:
def evaluate_ensemble(X, y, ensemble, rskf):
    """Evaluate an ensemble pipeline with cross_validate."""
    scoring = ['accuracy', 'f1_weighted', 'roc_auc']
    cv_results = cross_validate(ensemble, X, y, cv=rskf, scoring=scoring,
                                 return_train_score=False, n_jobs=-1)
    acc = cv_results['test_accuracy']
    f1 = cv_results['test_f1_weighted']
    auc = cv_results['test_roc_auc']
    return acc, f1, auc


def build_and_evaluate_ensembles(X, y, selected_models, stability_df, fold_scores, dataset_name):
    """Build all ensemble methods and evaluate them. Returns master results DataFrame and fold scores dict."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    all_classifiers = get_base_classifiers()
    
    # Collect all fold scores for box plots
    all_fold_scores = dict(fold_scores)  # copy individual scores
    
    results = []
    
    # 1. Individual classifiers — reuse from stability_df
    for _, row in stability_df.iterrows():
        results.append({
            'Method': f"{row['Model']} (default)",
            'Mean Acc': row['Mean Acc'], 'Std Acc': row['Std Acc'],
            'Mean F1': row['Mean F1'], 'Std F1': row['Std F1'],
            'Mean AUC': row['Mean AUC'], 'Std AUC': row['Std AUC']
        })
    
    # 2. Full Soft Voting Ensemble
    print(f'  Evaluating Full Soft Voting...')
    full_estimators = [(name, clf) for name, clf in all_classifiers.items()]
    full_voting = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('ensemble', VotingClassifier(estimators=full_estimators, voting='soft'))
    ])
    acc, f1, auc = evaluate_ensemble(X, y, full_voting, rskf)
    all_fold_scores['Full Soft Voting'] = auc
    results.append({
        'Method': 'Full Soft Voting',
        'Mean Acc': acc.mean(), 'Std Acc': acc.std(),
        'Mean F1': f1.mean(), 'Std F1': f1.std(),
        'Mean AUC': auc.mean(), 'Std AUC': auc.std()
    })
    
    # 3. Full Stacking Ensemble
    print(f'  Evaluating Full Stacking...')
    full_stacking = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('ensemble', StackingClassifier(
            estimators=full_estimators,
            final_estimator=LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
            cv=5
        ))
    ])
    acc, f1, auc = evaluate_ensemble(X, y, full_stacking, rskf)
    all_fold_scores['Full Stacking'] = auc
    results.append({
        'Method': 'Full Stacking',
        'Mean Acc': acc.mean(), 'Std Acc': acc.std(),
        'Mean F1': f1.mean(), 'Std F1': f1.std(),
        'Mean AUC': auc.mean(), 'Std AUC': auc.std()
    })
    
    # 4. Pruned Soft Voting Ensemble (default params)
    print(f'  Evaluating Pruned Soft Voting (default)...')
    pruned_estimators = [(name, all_classifiers[name]) for name in selected_models]
    pruned_voting = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('ensemble', VotingClassifier(estimators=pruned_estimators, voting='soft'))
    ])
    acc, f1, auc = evaluate_ensemble(X, y, pruned_voting, rskf)
    all_fold_scores['Pruned Soft Voting (default)'] = auc
    results.append({
        'Method': 'Pruned Soft Voting (default)',
        'Mean Acc': acc.mean(), 'Std Acc': acc.std(),
        'Mean F1': f1.mean(), 'Std F1': f1.std(),
        'Mean AUC': auc.mean(), 'Std AUC': auc.std()
    })
    
    return results, all_fold_scores


print('=== Stage 3 — Heart Disease ===')
heart_results, heart_all_fold_scores = build_and_evaluate_ensembles(
    heart_X, heart_y, heart_selected, heart_stability_df, heart_fold_scores, 'heart')

print('\n=== Stage 3 — Pima Diabetes ===')
pima_results, pima_all_fold_scores = build_and_evaluate_ensembles(
    pima_X, pima_y, pima_selected, pima_stability_df, pima_fold_scores, 'pima')

=== Stage 3 — Heart Disease ===
  Evaluating Full Soft Voting...
  Evaluating Full Stacking...
  Evaluating Pruned Soft Voting (default)...

=== Stage 3 — Pima Diabetes ===
  Evaluating Full Soft Voting...
  Evaluating Full Stacking...
  Evaluating Pruned Soft Voting (default)...


## 7. Stage 4 — Hyperparameter Tuning (Pruned Models Only)

Tuning happens **after** pruning. Pruning was done on neutral default params to avoid bias.  
Tuning here serves only to enhance the final pruned ensemble.

In [9]:
PARAM_GRIDS = {
    'LR': {'clf__C': [0.01, 0.1, 1, 10]},
    'SVM': {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 'auto']},
    'kNN': {'clf__n_neighbors': [3, 5, 7, 11]},
    'RF': {'clf__n_estimators': [100, 200], 'clf__max_depth': [None, 5, 10]},
    'XGBoost': {'clf__n_estimators': [100, 200], 'clf__learning_rate': [0.05, 0.1]}
}


def tune_and_build_pruned_ensemble(X, y, selected_models, dataset_name):
    """Tune each selected model via GridSearchCV, then build tuned pruned ensemble."""
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    all_classifiers = get_base_classifiers()
    
    tuned_classifiers = {}
    print(f'\n=== Hyperparameter Tuning — {dataset_name.upper()} ===')
    
    for name in selected_models:
        clf = all_classifiers[name]
        pipe = make_pipeline(clf)
        grid = PARAM_GRIDS[name]
        
        gs = GridSearchCV(pipe, grid, cv=3, scoring='roc_auc', n_jobs=-1)
        gs.fit(X, y)
        
        best_clf = gs.best_estimator_.named_steps['clf']
        tuned_classifiers[name] = best_clf
        print(f'  {name}: best params = {gs.best_params_}, best AUC = {gs.best_score_:.4f}')
    
    # Build tuned pruned voting ensemble
    tuned_estimators = [(name, tuned_classifiers[name]) for name in selected_models]
    tuned_pruned_voting = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('ensemble', VotingClassifier(estimators=tuned_estimators, voting='soft'))
    ])
    
    # Evaluate
    print(f'  Evaluating Pruned Soft Voting (tuned)...')
    scoring = ['accuracy', 'f1_weighted', 'roc_auc']
    cv_results = cross_validate(tuned_pruned_voting, X, y, cv=rskf, scoring=scoring,
                                 return_train_score=False, n_jobs=-1)
    acc = cv_results['test_accuracy']
    f1 = cv_results['test_f1_weighted']
    auc = cv_results['test_roc_auc']
    
    result = {
        'Method': 'Pruned Soft Voting (tuned)',
        'Mean Acc': acc.mean(), 'Std Acc': acc.std(),
        'Mean F1': f1.mean(), 'Std F1': f1.std(),
        'Mean AUC': auc.mean(), 'Std AUC': auc.std()
    }
    
    return result, auc


heart_tuned_result, heart_tuned_auc = tune_and_build_pruned_ensemble(
    heart_X, heart_y, heart_selected, 'heart')

pima_tuned_result, pima_tuned_auc = tune_and_build_pruned_ensemble(
    pima_X, pima_y, pima_selected, 'pima')


=== Hyperparameter Tuning — HEART ===
  LR: best params = {'clf__C': 0.01}, best AUC = 0.9046
  SVM: best params = {'clf__C': 0.1, 'clf__gamma': 'scale'}, best AUC = 0.9001
  RF: best params = {'clf__max_depth': 5, 'clf__n_estimators': 200}, best AUC = 0.9059
  Evaluating Pruned Soft Voting (tuned)...

=== Hyperparameter Tuning — PIMA ===
  LR: best params = {'clf__C': 1}, best AUC = 0.8371
  SVM: best params = {'clf__C': 1, 'clf__gamma': 'scale'}, best AUC = 0.8259
  Evaluating Pruned Soft Voting (tuned)...


## 8. Final Results Table

In [10]:
def build_final_table(results_list, tuned_result, dataset_name):
    """Combine all results into master DataFrame, highlight best values."""
    all_results = results_list + [tuned_result]
    df = pd.DataFrame(all_results)
    df = df.set_index('Method')
    
    df.to_csv(f'final_results_{dataset_name}.csv')
    
    print(f'\n{"=" * 80}')
    print(f'FINAL RESULTS — {dataset_name.upper()}')
    print(f'{"=" * 80}')
    
    # Format for display
    display_df = df.copy()
    for col in df.columns:
        display_df[col] = df[col].map(lambda x: f'{x:.4f}')
    
    # Highlight best (max for Mean, min for Std)
    for col in df.columns:
        if 'Std' in col:
            best_idx = df[col].idxmin()
        else:
            best_idx = df[col].idxmax()
        display_df.loc[best_idx, col] = display_df.loc[best_idx, col] + ' *'
    
    print(display_df.to_string())
    print('\n(* = best value in column)')
    return df


heart_final_df = build_final_table(heart_results, heart_tuned_result, 'heart')
pima_final_df = build_final_table(pima_results, pima_tuned_result, 'pima')


FINAL RESULTS — HEART
                              Mean Acc   Std Acc   Mean F1    Std F1  Mean AUC   Std AUC
Method                                                                                  
LR (default)                    0.8375    0.0424    0.8365    0.0426    0.9059    0.0391
SVM (default)                   0.8342    0.0382    0.8331    0.0385    0.8990    0.0378
kNN (default)                   0.8303    0.0375    0.8295    0.0379    0.8907    0.0387
RF (default)                    0.8315    0.0496    0.8302    0.0502    0.9047    0.0369
XGBoost (default)               0.8038    0.0464    0.8027    0.0471    0.8787    0.0358
Full Soft Voting                0.8322  0.0358 *    0.8312  0.0360 *    0.9096    0.0364
Full Stacking                   0.8368    0.0395    0.8359    0.0398    0.9097    0.0374
Pruned Soft Voting (default)    0.8342    0.0430    0.8333    0.0432  0.9101 *    0.0371
Pruned Soft Voting (tuned)    0.8388 *    0.0395  0.8377 *    0.0400    0.9100  0.0353 

## 9. Visualizations

In [ ]:
# --- Plot 1: Bar chart — ROC-AUC mean + error bars ---
def plot_auc_comparison(final_df, dataset_name):
    fig, ax = plt.subplots(figsize=(12, 6))
    methods = final_df.index.tolist()
    means = final_df['Mean AUC'].values
    stds = final_df['Std AUC'].values
    
    colors = sns.color_palette('viridis', len(methods))
    bars = ax.bar(range(len(methods)), means, yerr=stds, capsize=4,
                  color=colors, edgecolor='black', linewidth=0.5)
    
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('ROC-AUC')
    ax.set_title(f'ROC-AUC Comparison — {dataset_name.upper()}')
    ax.set_ylim(0.5, 1.0)
    plt.tight_layout()
    plt.savefig(f'plot_auc_comparison_{dataset_name}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved plot_auc_comparison_{dataset_name}.png')

plot_auc_comparison(heart_final_df, 'heart')
plot_auc_comparison(pima_final_df, 'pima')

Saved plot_auc_comparison_heart.png
Saved plot_auc_comparison_pima.png


In [ ]:
# --- Plot 2: Heatmap — stability scores across both datasets ---
model_names = ['LR', 'SVM', 'kNN', 'RF', 'XGBoost']
heart_stab = heart_stability_df.set_index('Model')['Stability Score']
pima_stab = pima_stability_df.set_index('Model')['Stability Score']

heatmap_df = pd.DataFrame({
    'Heart Disease': [heart_stab[m] for m in model_names],
    'Pima Diabetes': [pima_stab[m] for m in model_names]
}, index=model_names)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='gray')
ax.set_title('Stability Scores (Mean AUC / Std AUC) Across Datasets')
ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig('plot_stability_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved plot_stability_heatmap.png')

Saved plot_stability_heatmap.png


In [ ]:
# --- Plot 3: Box plots — AUC distribution across folds ---
def plot_boxplot(fold_scores, tuned_auc, dataset_name):
    # Add tuned scores
    all_scores = dict(fold_scores)
    all_scores['Pruned Soft Voting (tuned)'] = tuned_auc
    
    # Order to match final results table
    order = ['LR', 'SVM', 'kNN', 'RF', 'XGBoost',
             'Full Soft Voting', 'Full Stacking',
             'Pruned Soft Voting (default)', 'Pruned Soft Voting (tuned)']
    
    data = []
    labels = []
    for name in order:
        if name in all_scores:
            data.append(all_scores[name])
            labels.append(name)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    bp = ax.boxplot(data, patch_artist=True, notch=True)
    
    colors = sns.color_palette('viridis', len(labels))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('ROC-AUC')
    ax.set_title(f'ROC-AUC Distribution Across Folds — {dataset_name.upper()}')
    plt.tight_layout()
    plt.savefig(f'plot_boxplot_{dataset_name}.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved plot_boxplot_{dataset_name}.png')

plot_boxplot(heart_all_fold_scores, heart_tuned_auc, 'heart')
plot_boxplot(pima_all_fold_scores, pima_tuned_auc, 'pima')

Saved plot_boxplot_heart.png
Saved plot_boxplot_pima.png


## 10. Output Files Summary

In [ ]:
import os

expected_files = [
    'heart_disease_clean.csv',
    'pima_diabetes_clean.csv',
    'stability_results_heart.csv',
    'stability_results_pima.csv',
    'final_results_heart.csv',
    'final_results_pima.csv',
    'plot_auc_comparison_heart.png',
    'plot_auc_comparison_pima.png',
    'plot_stability_heatmap.png',
    'plot_boxplot_heart.png',
    'plot_boxplot_pima.png'
]

print('=== Generated Files ===')
for f in expected_files:
    exists = os.path.isfile(f)
    size = os.path.getsize(f) if exists else 0
    status = f'{size:,} bytes' if exists else 'MISSING'
    print(f'  {f:45s} {status}')

print('\nDone! All stages completed successfully.')

=== Generated Files ===
  heart_disease_clean.csv                       12,475 bytes
  pima_diabetes_clean.csv                       23,290 bytes
  stability_results_heart.csv                   778 bytes
  stability_results_pima.csv                    773 bytes
  final_results_heart.csv                       1,271 bytes
  final_results_pima.csv                        1,278 bytes
  plot_auc_comparison_heart.png                 66,591 bytes
  plot_auc_comparison_pima.png                  66,743 bytes
  plot_stability_heatmap.png                    46,666 bytes
  plot_boxplot_heart.png                        100,727 bytes
  plot_boxplot_pima.png                         97,877 bytes

Done! All stages completed successfully.
